In [2]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_distances
from skbio.stats.distance import mantel, DistanceMatrix

data

In [3]:
df = pd.read_pickle('aft_df_with_extracted_motif3.pkl')

language tree

In [4]:
LANG = {
    'Germany':            ('IE', 'Germanic', 'West', 'German'),
    'Austria':            ('IE', 'Germanic', 'West', 'German'),
    'Switzerland':        ('IE', 'Germanic', 'West', 'German'),
    'England':            ('IE', 'Germanic', 'West', 'English'),
    'USA':                ('IE', 'Germanic', 'West', 'English'),
    'Jamaica':            ('IE', 'Germanic', 'West', 'English'),
    'Netherlands':        ('IE', 'Germanic', 'West', 'Dutch'),
    'Flanders':           ('IE', 'Germanic', 'West', 'Dutch'),
    'South Africa':       ('IE', 'Germanic', 'West', 'Afrikaans'),
    'Norway':             ('IE', 'Germanic', 'North', 'Norwegian'),
    'Denmark':            ('IE', 'Germanic', 'North', 'Danish'),
    'Sweden':             ('IE', 'Germanic', 'North', 'Swedish'),
    'Iceland':            ('IE', 'Germanic', 'North', 'Icelandic'),
    'Italy':              ('IE', 'Romance', 'Romance', 'Italian'),
    'France':             ('IE', 'Romance', 'Romance', 'French'),
    'Spain':              ('IE', 'Romance', 'Romance', 'Spanish'),
    'Portugal':           ('IE', 'Romance', 'Romance', 'Portuguese'),
    'Romania':            ('IE', 'Romance', 'Romance', 'Romanian'),
    'Russia':             ('IE', 'Slavic', 'East', 'Russian'),
    'Ukraine':            ('IE', 'Slavic', 'East', 'Ukrainian'),
    'Poland':             ('IE', 'Slavic', 'West', 'Polish'),
    'Moravia':            ('IE', 'Slavic', 'West', 'Czech'),
    'Bohemia':            ('IE', 'Slavic', 'West', 'Czech'),
    'Serbia':             ('IE', 'Slavic', 'South', 'Serbian'),
    'Scotland':           ('IE', 'Celtic', 'Goidelic', 'Gaelic'),
    'Ireland':            ('IE', 'Celtic', 'Goidelic', 'Irish'),
    'Wales':              ('IE', 'Celtic', 'Brythonic', 'Welsh'),
    'India':              ('IE', 'IndoIranian', 'IndoAryan', 'Hindi'),
    'Pakistan':           ('IE', 'IndoIranian', 'IndoAryan', 'Urdu'),
    'Sri Lanka':          ('IE', 'IndoIranian', 'IndoAryan', 'Sinhala'),
    'Kashmir':            ('IE', 'IndoIranian', 'IndoAryan', 'Kashmiri'),
    'Lithuania':          ('IE', 'BaltoSlavic', 'Baltic', 'Lithuanian'),
    'Greece':             ('IE', 'Hellenic', 'Hellenic', 'Greek'),
    'Finland':            ('Uralic', 'FinnoUgric', 'Finnic', 'Finnish'),
    'Hungary':            ('Uralic', 'FinnoUgric', 'Ugric', 'Hungarian'),
    'Turkey':             ('Turkic', 'Turkic', 'Turkic', 'Turkish'),
    'Philippines':        ('Austronesian', 'Malayo-Polynesian', 'Philippine', 'Tagalog'),
    'Malaya':             ('Austronesian', 'Malayo-Polynesian', 'Malayic', 'Malay'),
    'Basque':             ('Isolate', 'Basque', 'Basque', 'Basque'),
    'Georgia':            ('Kartvelian', 'Kartvelian', 'Kartvelian', 'Georgian'),
    'Japan':              ('Japonic', 'Japonic', 'Japonic', 'Japanese'),
    'Tibet':              ('SinoTibetan', 'Tibetic', 'Tibetic', 'Tibetan'),
    'Palestine':          ('AfroAsiatic', 'Semitic', 'Semitic', 'Arabic'),
    'Korea':              ('Koreanic', 'Koreanic', 'Koreanic', 'Korean'),
    'China':              ('SinoTibetan', 'Sinitic', 'Sinitic', 'Chinese')
}

calculate distance

In [5]:
def lingdist(a, b):
    if a == b:
        return 0
    fa, ba, sa, la = LANG[a]
    fb, bb, sb, lb = LANG[b]
    if la == lb:
        return 0
    if sa == sb:
        return 1
    if ba == bb:
        return 2
    if fa == fb:
        return 3
    return 4

verify how many clear regions we have

In [6]:
provs = [p for p in LANG if (df['provenance'] == p).sum() >= 3]
print(f"Using {len(provs)} provenances:", provs)

Using 45 provenances: ['Germany', 'Austria', 'Switzerland', 'England', 'USA', 'Jamaica', 'Netherlands', 'Flanders', 'South Africa', 'Norway', 'Denmark', 'Sweden', 'Iceland', 'Italy', 'France', 'Spain', 'Portugal', 'Romania', 'Russia', 'Ukraine', 'Poland', 'Moravia', 'Bohemia', 'Serbia', 'Scotland', 'Ireland', 'Wales', 'India', 'Pakistan', 'Sri Lanka', 'Kashmir', 'Lithuania', 'Greece', 'Finland', 'Hungary', 'Turkey', 'Philippines', 'Malaya', 'Basque', 'Georgia', 'Japan', 'Tibet', 'Palestine', 'Korea', 'China']


In [7]:
total_rows_for_provs = df[df['provenance'].isin(provs)].shape[0]
print(f"Using a total of {total_rows_for_provs} rows.")

Using a total of 885 rows.


## Our Motifs

In [8]:
motif_text = {}
motif_sets = {}
for p in provs:
    sub = df[df['provenance'] == p]
    motifs = []
    motif_set = set()
    for ms in sub['extracted_motifs']:
        if isinstance(ms, list):
            motifs.extend(ms)
            for m in ms:
                motif_set.add(m.strip().lower())
    motif_text[p] = ' '.join(motifs)
    motif_sets[p] = motif_set

lingustic distance matrix

In [9]:
n = len(provs)
ling = np.zeros((n, n))
for i, j in combinations(range(n), 2):
    d = lingdist(provs[i], provs[j])
    ling[i, j] = ling[j, i] = d

tf idf matrix using TfidfVectorizer and cosine distance

In [10]:
tfidf = TfidfVectorizer(stop_words='english')
X = tfidf.fit_transform([motif_text[p] for p in provs])
motif_tfidf_dist = cosine_distances(X)

jaccard distance

In [11]:
motif_jaccard = np.zeros((n, n))
for i, j in combinations(range(n), 2):
    a, b = motif_sets[provs[i]], motif_sets[provs[j]]
    union = len(a | b)
    inter = len(a & b)
    d = 1 - inter / union if union else 1
    motif_jaccard[i, j] = motif_jaccard[j, i] = d

In [12]:
ling_dm = DistanceMatrix(ling, provs)
tfidf_dm = DistanceMatrix(motif_tfidf_dist, provs)
jaccard_dm = DistanceMatrix(motif_jaccard, provs)

In [13]:
print("\n--- Mantel: linguistic distance vs motif TF-IDF distance ---")
r, p, n_ = mantel(ling_dm, tfidf_dm, method='spearman', permutations=9999)
print(f"r={r:.4f}, p={p:.4f}, n={n_}")

print("\n--- Mantel: linguistic distance vs motif Jaccard distance ---")
r, p, n_ = mantel(ling_dm, jaccard_dm, method='spearman', permutations=9999)
print(f"r={r:.4f}, p={p:.4f}, n={n_}")


--- Mantel: linguistic distance vs motif TF-IDF distance ---
r=0.2889, p=0.0178, n=45

--- Mantel: linguistic distance vs motif Jaccard distance ---
r=0.2139, p=0.0172, n=45


## Story text

In [14]:
story_text = {}
word_sets = {}
for p in provs:
    sub = df[df['provenance'] == p]
    texts = []
    words = set()
    for t in sub['text']:
        if isinstance(t, str):
            texts.append(t)
            for w in t.lower().split():
                words.add(w.strip('.,;:!?"\'-()[]'))
    story_text[p] = ' '.join(texts)
    word_sets[p] = words

In [15]:
tfidf = TfidfVectorizer(stop_words='english')
X = tfidf.fit_transform([story_text[p] for p in provs])
story_tfidf_dist = cosine_distances(X)

In [16]:
story_jaccard = np.zeros((n, n))
for i, j in combinations(range(n), 2):
    a, b = word_sets[provs[i]], word_sets[provs[j]]
    union = len(a | b)
    inter = len(a & b)
    d = 1 - inter / union if union else 1
    story_jaccard[i, j] = story_jaccard[j, i] = d

In [17]:
tfidf_dm = DistanceMatrix(story_tfidf_dist, provs)
jaccard_dm = DistanceMatrix(story_jaccard, provs)

In [18]:
print("\n--- Mantel: linguistic distance vs story-text TF-IDF distance ---")
r, p, n_ = mantel(ling_dm, tfidf_dm, method='spearman', permutations=9999)
print(f"r={r:.4f}, p={p:.4f}, n={n_}")

print("\n--- Mantel: linguistic distance vs story-text Jaccard distance ---")
r, p, n_ = mantel(ling_dm, jaccard_dm, method='spearman', permutations=9999)
print(f"r={r:.4f}, p={p:.4f}, n={n_}")


--- Mantel: linguistic distance vs story-text TF-IDF distance ---
r=0.2889, p=0.0184, n=45

--- Mantel: linguistic distance vs story-text Jaccard distance ---
r=-0.0226, p=0.8119, n=45
